# Text-to-SQL Backend — Google Colab
This notebook loads Qwen2.5-1.5B-Instruct + the LoRA adapter, attaches the model to FastAPI in the same Python process, creates a demo SQLite database, and demonstrates JSON caching of database metadata/schema/retrieval context.


In [ ]:
!pip -q install fastapi uvicorn python-multipart transformers peft accelerate pyngrok

In [ ]:
# Upload/copy the repository into /content, then run from its parent directory.
%cd /content/NLP-Based-Text-to-SQL-System

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_id = 'Qwen/Qwen2.5-1.5B-Instruct'
adapter_id = 'D3v1k4/qwen-nlp-text-to-sql'

tokenizer = AutoTokenizer.from_pretrained(base_id)
base_model = AutoModelForCausalLM.from_pretrained(base_id)
model = PeftModel.from_pretrained(base_model, adapter_id)
print('Model loaded')

In [ ]:
from backend.pipeline import pipeline
pipeline.model.attach(model, tokenizer)
print('Backend connected to model!')

In [ ]:
from pathlib import Path
import sqlite3

db_dir = Path('databases')
db_dir.mkdir(exist_ok=True)
db = db_dir / 'demo.sqlite'
con = sqlite3.connect(db)
con.execute('DROP TABLE IF EXISTS students')
con.execute('CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, department TEXT NOT NULL, year INTEGER, marks INTEGER)')
rows = [(1,'Arun','CSE',4,92),(2,'Rahul','CSE',3,78),(3,'Priya','ECE',4,88),(4,'Karthik','CSE',2,65),(5,'Anjali','EEE',3,95),(6,'Vikram','ECE',2,72),(7,'Sneha','CSE',4,84),(8,'Rohit','ME',3,61),(9,'Divya','CSE',3,91),(10,'Aditya','ECE',4,76)]
con.executemany('INSERT INTO students VALUES (?,?,?,?,?)', rows)
con.commit(); con.close()
print(db)

In [ ]:
from backend.db import get_cache_status, get_database_cache

cache = get_database_cache('demo')
print('First access cache hit:', cache['cache_hit'])
print('Cache directory:', cache['cache_dir'])
print('SHA-256:', cache['metadata']['sha256'])


In [ ]:
import threading, uvicorn
from backend.app import app
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
print('FastAPI started on port 8000')

In [ ]:
import requests
print(requests.get('http://127.0.0.1:8000/health').json())
print(requests.get('http://127.0.0.1:8000/databases').json())

In [ ]:
print(requests.get('http://127.0.0.1:8000/cache', params={'database':'demo'}).json())


## Optional: ngrok
Use your own ngrok account. Do **not** hard-code the authtoken in this notebook before committing it. Store it in a secret or enter it interactively.

In [ ]:
from pyngrok import ngrok
# ngrok.set_auth_token('PASTE_TOKEN_ONLY_IN_RUNTIME — DO NOT COMMIT')
# tunnel = ngrok.connect(8000, 'http')
# print(tunnel.public_url)